# 04 — Pipeline de ML: detección de fraude (`bank_transactions.csv`)

**Objetivo:** cerrar la progresión de Spark con un `Pipeline` de MLlib completo
(feature engineering → escalado → modelo), el puente directo hacia la sesión de
"Ingeniería de features a escala" del track de Maestría.

**Dataset:** `bank_transactions.csv` (~7.5 GB), mismo dataset que
`03_spark_sql.ipynb` — la columna `is_suspicious` ya viene lista para usar como label.

> **Nota de diseño:** la columna `suspicious_pattern` NO se usa como feature a
> propósito — describe *el tipo* de patrón sospechoso detectado, así que solo tiene
> un valor no nulo cuando `is_suspicious = true`. Usarla como feature sería fuga de
> información (el modelo "vería" la respuesta disfrazada de pregunta).

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import Imputer, StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

spark = SparkSession.builder.appName("04_pipeline_ml").getOrCreate()

RUTA = "gs://<TU-BUCKET>/raw/bank_transactions/bank_transactions.csv"

In [ ]:
df = spark.read.csv(RUTA, header=True, inferSchema=True)
df = df.withColumn("is_suspicious", F.col("is_suspicious").cast("int"))
df = df.withColumn("hora_del_dia", F.hour("timestamp"))
df.select("amount", "currency", "hora_del_dia", "is_suspicious").show(5)

## 1. Imputación de nulos en columnas numéricas

In [ ]:
imputer = Imputer(
    inputCols=["amount", "hora_del_dia"],
    outputCols=["amount_imputado", "hora_del_dia_imputada"],
)

## 2. Encoding de la variable categórica (`currency`)

In [ ]:
indexer = StringIndexer(inputCol="currency", outputCol="currency_idx", handleInvalid="keep")
encoder = OneHotEncoder(inputCols=["currency_idx"], outputCols=["currency_ohe"])

## 3. Ensamblado del vector de features

In [ ]:
assembler = VectorAssembler(
    inputCols=["amount_imputado", "hora_del_dia_imputada", "currency_ohe"],
    outputCol="features_raw",
)

## 4. Escalado

In [ ]:
scaler = StandardScaler(inputCol="features_raw", outputCol="features")

## 5. Modelo

In [ ]:
lr = LogisticRegression(featuresCol="features", labelCol="is_suspicious")

pipeline = Pipeline(stages=[imputer, indexer, encoder, assembler, scaler, lr])

## 6. Split y entrenamiento

In [ ]:
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)
modelo = pipeline.fit(train_df)

## 7. Evaluación

In [ ]:
predicciones = modelo.transform(test_df)
evaluador = BinaryClassificationEvaluator(labelCol="is_suspicious", metricName="areaUnderROC")
auc = evaluador.evaluate(predicciones)
print(f"=== AUC en test: {auc:.4f} ===")

## 8. Guardar el pipeline entrenado

Reproducible — se puede cargar directo en la sesión de model serving (Sesión 8 de
Maestría) para envolverlo en un endpoint.

In [ ]:
modelo.write().overwrite().save("gs://<TU-BUCKET>/modelos/fraude_bank_transactions_pipeline")

In [ ]:
spark.stop()